# 🧭 Sequential Boundary Classifiers for Colonoscopy Localization
### VIM Polyp Dataset · Adjacent-Segment Binary Classifiers

**Idea:** Instead of a single multi-class model that tries to classify *all* colon
segments at once, train a **separate binary classifier for each pair of
anatomically adjacent segments** (e.g. `cecum vs ascending`, `ascending vs
transverse`, `transverse vs descending`, ...).

During a real colonoscopy withdrawal, the scope moves through the colon in a
(mostly) fixed anatomical order. A bank of "boundary detectors" — one per
adjacent pair — can each specialize in telling two *neighboring* segments
apart, which is usually an easier problem than telling all segments apart at
once (neighboring segments tend to look more visually similar to each other
than to segments elsewhere in the colon, so a model that only has to solve
that specific pairwise problem can focus its capacity there). Chaining the
boundary detectors together then gives a simple way to estimate roughly
*where the scope is* in the procedure.

**Pipeline Overview:**
1. Environment Setup
2. Configuration — anatomical order of segments (**edit this for your data**)
3. Frame Extraction (same extraction/cleaning logic as the main notebook)
4. Build Adjacent Pairs
5. Per-Pair Dataset Builder (filter → balance → stratified split → YOLO folders)
6. Train One Binary Classifier per Adjacent Pair
7. Evaluate Each Boundary Classifier
8. Summary Dashboard Across All Boundaries
9. (Optional) Chaining Boundary Classifiers into a Position Estimate

> This notebook assumes the same raw video layout as `VIM_YOLO_Localization.ipynb`
> (filenames encoding the anatomical segment at index 5 when split on `-`).
> If your naming convention differs, adjust `extract_segment_from_filename()` in
> Section 3.

---


## 1. Environment Setup

In [12]:
# ─── Imports ──────────────────────────────────────────────────────────────
import os
import cv2
import shutil
import random
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score

from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print(f"✅ Using device: {DEVICE}")


✅ Using device: 0


## 2. Configuration

**Edit the values in this cell for your dataset before running anything else.**

In [13]:
# LOCAL_DATA_DIR = r"D:\Marina\colonVideosWithLabels"  
LOCAL_DATA_DIR = r"D:\Work\Nebras\colonVideosWithLabels"

In [14]:
# ─── Paths ────────────────────────────────────────────────────────────────
 # same raw video folder as the main notebook

ROOT = Path('vim_polyp_pairwise')
TEMP_FRAMES_DIR = ROOT / 'data' / 'temp_frames'
PAIRWISE_DATA_DIR = ROOT / 'pairwise_data'
RESULTS = ROOT / 'results'
for d in (TEMP_FRAMES_DIR, PAIRWISE_DATA_DIR, RESULTS):
    d.mkdir(parents=True, exist_ok=True)

# Segments to exclude entirely from this experiment (same as the main notebook)
CLASSES_TO_DROP = {'anastomosis', 'rectosigmoid', 'colon'}

# Example matching the request: cecum -> ascending -> transverse
ANATOMICAL_ORDER = [
    'cecum',
    'ascending',
    'transverse',
    'descending',
    'sigmoid',
    'rectum'
]

print("Anatomical order configured:")
for i, seg in enumerate(ANATOMICAL_ORDER):
    print(f"  {i}. {seg}")

# ─── Balancing strategy for each pairwise dataset ────────────────────────────
# 'undersample' -> cap both classes in a pair to the smaller class's count
# 'oversample'   -> duplicate the smaller class up to the larger class's count
PAIR_BALANCE_STRATEGY = 'undersample'

# ─── Frame extraction settings ───────────────────────────────────────────────
FRAMES_PER_SECOND = 1  # how many frames per second of video to sample

# Ensure the notebook has default values for later training/augmentation cells.
CLASS_NAMES = []
TRAIN_CFG = {}

Anatomical order configured:
  0. cecum
  1. ascending
  2. transverse
  3. descending
  4. sigmoid
  5. rectum


## 3. Frame Extraction & Cleaning

Same artifact-filtering logic as the main notebook — pitch-black frames and solid color/blue-screen artifacts are skipped.

In [15]:
def is_unwanted_frame(frame, brightness_thresh=15, variance_thresh=5):
    """Returns True if the frame is pitch black, a solid blue screen, or a blank artifact."""
    means, stddevs = cv2.meanStdDev(frame)
    if max(means) < brightness_thresh:
        return True
    if max(stddevs) < variance_thresh:
        return True
    blue_mean, green_mean, red_mean = means[0][0], means[1][0], means[2][0]
    if blue_mean > 150 and green_mean < 40 and red_mean < 40:
        return True
    return False


def extract_segment_from_filename(filename):
    """
    Extracts the anatomical segment label from a video filename.
    Assumes the VIM Polyp naming convention: segment is token index 5 when
    the filename stem is split on '-'. Adjust this if your filenames differ.
    """
    stem = os.path.splitext(filename)[0]
    parts = stem.split('-')
    if len(parts) > 5:
        return parts[5].lower().strip()
    return None


# ─── Scan raw video directory ────────────────────────────────────────────────
all_video_files = []
if os.path.exists(LOCAL_DATA_DIR):
    for entry in os.scandir(LOCAL_DATA_DIR):
        if not entry.is_file():
            continue
        filename = entry.name
        if filename.startswith('.') or filename.startswith('._'):
            continue
        if not filename.lower().endswith(('.avi', '.mp4', '.mkv')):
            continue

        segment = extract_segment_from_filename(filename)
        if segment is None or segment in CLASSES_TO_DROP:
            continue
        # Only keep videos for segments we actually care about (in ANATOMICAL_ORDER)
        if segment not in ANATOMICAL_ORDER:
            continue

        all_video_files.append((Path(entry.path), segment))

    print(f"✅ Found {len(all_video_files)} relevant videos "
          f"(segments restricted to ANATOMICAL_ORDER).")
else:
    print(f"❌ Local data directory not found at: {LOCAL_DATA_DIR}")

# ─── Extract clean frames ─────────────────────────────────────────────────────
all_pairs = []   # list of (frame_name, segment_label)
skipped_artifacts = 0

if all_video_files:
    for video_path, segment in tqdm(all_video_files, desc="Extracting clean frames"):
        video_id = video_path.stem
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            continue

        fps = cap.get(cv2.CAP_PROP_FPS) or 30
        frame_interval = max(1, int(fps / FRAMES_PER_SECOND))
        frame_count = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if frame_count % frame_interval == 0:
                if is_unwanted_frame(frame):
                    skipped_artifacts += 1
                    frame_count += 1
                    continue

                frame_name = f"{video_id}_frame_{frame_count:05d}.jpg"
                frame_path = TEMP_FRAMES_DIR / frame_name
                cv2.imwrite(str(frame_path), frame)
                all_pairs.append((frame_name, segment))

            frame_count += 1
        cap.release()

    print(f"\n✅ Total valid extracted frames: {len(all_pairs)}")
    print(f"🗑️  Skipped {skipped_artifacts} unwanted artifact/blank frames.")

    df_all = pd.DataFrame(all_pairs, columns=['frame_name', 'segment'])
    print("\n📊 Frame counts per segment:")
    print(df_all['segment'].value_counts().reindex(ANATOMICAL_ORDER).to_string())
else:
    print("⚠️  No video files to process — check LOCAL_DATA_DIR and ANATOMICAL_ORDER.")
    df_all = pd.DataFrame(columns=['frame_name', 'segment'])


✅ Found 118 relevant videos (segments restricted to ANATOMICAL_ORDER).


Extracting clean frames: 100%|██████████| 118/118 [11:19<00:00,  5.76s/it]


✅ Total valid extracted frames: 9023
🗑️  Skipped 744 unwanted artifact/blank frames.

📊 Frame counts per segment:
segment
cecum           30
ascending      946
transverse     166
descending    2619
sigmoid       2827
rectum        2435


## 4. Build Adjacent Pairs

Each consecutive pair in `ANATOMICAL_ORDER` becomes one binary classification problem — this is the direct implementation of "a model for cecum vs ascending, another for ascending vs transverse", etc.

In [16]:
PAIR_LIST = list(zip(ANATOMICAL_ORDER[:-1], ANATOMICAL_ORDER[1:]))

print(f"{len(PAIR_LIST)} adjacent-segment boundary classifiers will be trained:")
for i, (a, b) in enumerate(PAIR_LIST, 1):
    print(f"  {i}. {a}  vs  {b}")


5 adjacent-segment boundary classifiers will be trained:
  1. cecum  vs  ascending
  2. ascending  vs  transverse
  3. transverse  vs  descending
  4. descending  vs  sigmoid
  5. sigmoid  vs  rectum


## 5. Per-Pair Dataset Builder

For each pair, this: filters frames to just those two classes, balances the class counts, does a stratified train/val/test split, and lays the files out in the standard YOLO classification folder structure (`train/<class>/*.jpg`, etc.).

In [17]:
def build_pairwise_dataset(class_a, class_b, df_all, pairwise_root, seed=SEED,
                            balance_strategy=PAIR_BALANCE_STRATEGY,
                            test_size=0.30, val_of_temp=0.50):
    """
    Builds a balanced, stratified, YOLO-classification-format dataset folder
    for a single binary (class_a vs class_b) problem.
    Returns (dataset_dir, counts_dict) or (None, None) if there isn't enough data.
    """
    pair_name = f"{class_a}_vs_{class_b}"
    dataset_dir = pairwise_root / pair_name

    df_pair = df_all[df_all['segment'].isin([class_a, class_b])].copy()
    if df_pair.empty or df_pair['segment'].nunique() < 2:
        print(f"   ⚠️  Skipping {pair_name}: insufficient data for one or both classes.")
        return None, None

    counts_before = df_pair['segment'].value_counts()

    if balance_strategy == 'undersample':
        target_n = int(counts_before.min())
        df_balanced = pd.concat(
            [g.sample(n=target_n, random_state=seed) for _, g in df_pair.groupby('segment')],
            ignore_index=True,
        )
    elif balance_strategy == 'oversample':
        target_n = int(counts_before.max())
        df_balanced = pd.concat(
            [g.sample(n=target_n, replace=True, random_state=seed) for _, g in df_pair.groupby('segment')],
            ignore_index=True,
        )
    else:
        target_n = None
        df_balanced = df_pair.reset_index(drop=True)

    df_balanced = df_balanced.sample(frac=1, random_state=seed).reset_index(drop=True)

    # De-duplicate destination filenames for any oversampled repeats
    df_balanced['dup_idx'] = df_balanced.groupby('frame_name').cumcount()
    df_balanced['dest_frame_name'] = df_balanced.apply(
        lambda r: r['frame_name'] if r['dup_idx'] == 0
        else f"{Path(r['frame_name']).stem}_dup{r['dup_idx']}{Path(r['frame_name']).suffix}",
        axis=1
    )

    # ─── Stratified split ────────────────────────────────────────────────
    src_dest_pairs = list(zip(df_balanced['frame_name'], df_balanced['dest_frame_name']))
    labels = df_balanced['segment'].tolist()

    train_sd, tmp_sd, train_labels, tmp_labels = train_test_split(
        src_dest_pairs, labels, test_size=test_size, stratify=labels, random_state=seed)
    val_sd, test_sd, val_labels, test_labels = train_test_split(
        tmp_sd, tmp_labels, test_size=val_of_temp, stratify=tmp_labels, random_state=seed)

    splits = {
        'train': list(zip(train_sd, train_labels)),
        'val':   list(zip(val_sd,   val_labels)),
        'test':  list(zip(test_sd,  test_labels)),
    }

    # ─── Organize into YOLO classification folders ───────────────────────
    if dataset_dir.exists():
        shutil.rmtree(dataset_dir)

    for split_name, split_data in splits.items():
        for (src_name, dest_name), label in split_data:
            dest_dir = dataset_dir / split_name / label
            dest_dir.mkdir(parents=True, exist_ok=True)
            src_path = TEMP_FRAMES_DIR / src_name
            dest_path = dest_dir / dest_name
            if src_path.exists():
                shutil.copy2(str(src_path), str(dest_path))

    counts = {split_name: pd.Series([l for _, l in split_data]).value_counts().to_dict()
              for split_name, split_data in splits.items()}

    print(f"   ✅ {pair_name}: "
          f"train={len(splits['train'])}, val={len(splits['val'])}, test={len(splits['test'])} "
          f"(balance='{balance_strategy}', target/class={target_n})")

    return dataset_dir, counts


print("✅ build_pairwise_dataset() ready.")


✅ build_pairwise_dataset() ready.


## 6. Train One Binary Classifier per Adjacent Pair

Each boundary gets its own lightweight YOLOv8n-cls binary classifier. 

In [18]:
import logging
from ultralytics.utils import LOGGER

# Quiets Ultralytics' internal INFO-level spam (config dumps, "Scanning...",
# "Overriding model.yaml...", etc.) — keeps warnings/errors visible.
LOGGER.setLevel(logging.WARNING)

TRAIN_CFG_BINARY = dict(
    epochs        = 50,
    imgsz         = 224,
    batch         = 16,
    device        = DEVICE,
    optimizer     = 'AdamW',
    lr0           = 1e-3,
    lrf           = 0.01,
    warmup_epochs = 3,
    cos_lr        = True,
    fliplr        = 0.5,
    verbose       = False,   # suppresses the per-epoch table; keep final val summary
    plots         = False,   # skip auto-generated PR-curve/confusion pngs during train
    project       = str(RESULTS),
)

pairwise_models = {}
pairwise_dataset_dirs = {}
pairwise_counts = {}

def train_boundary_pair(class_a, class_b):
    """Builds the dataset + trains one boundary classifier. Safe to re-run alone."""
    pair_name = f"{class_a}_vs_{class_b}"
    print(f"▶ {pair_name}")

    dataset_dir, counts = build_pairwise_dataset(class_a, class_b, df_all, PAIRWISE_DATA_DIR)
    if dataset_dir is None:
        print(f"   ⏭️  skipped — insufficient data")
        return None

    pairwise_dataset_dirs[pair_name] = dataset_dir
    pairwise_counts[pair_name] = counts

    model = YOLO('yolov8n-cls.pt')
    model.train(data=str(dataset_dir), name=pair_name, **TRAIN_CFG_BINARY)
    pairwise_models[pair_name] = model

    print(f"   ✅ done — train={counts['train']}, val={counts['val']}, test={counts['test']}")
    return model

print(f"Ready. {len(PAIR_LIST)} pairs queued:", [f"{a}_vs_{b}" for a, b in PAIR_LIST])

Ready. 5 pairs queued: ['cecum_vs_ascending', 'ascending_vs_transverse', 'transverse_vs_descending', 'descending_vs_sigmoid', 'sigmoid_vs_rectum']


In [ ]:
# ─── Domain-Robustness Augmentation: CLAHE + Histogram Matching + Color Jitter ───
# Goal: reduce the model's reliance on VIM-Polyp's specific scope/illumination
# calibration, so it generalizes better to Kvasir/C3VD-style images at test time.

import cv2
import numpy as np
import random
import glob
from pathlib import Path
from tqdm.auto import tqdm

random.seed(SEED)
np.random.seed(SEED)

# ── 1. CLAHE illumination normalization ─────────────────────────────────────
# Normalizes local contrast/illumination so the model can't lean on VIM's
# specific brightness/contrast signature as a shortcut.
def apply_clahe(img_bgr, clip_limit=2.5, tile_grid_size=(8, 8)):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    l_eq = clahe.apply(l)
    lab_eq = cv2.merge((l_eq, a, b))
    return cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)


# ── 2. Manual histogram matching to a target-domain reference pool ──────────
# Matches per-channel intensity distribution of a VIM frame to a randomly
# sampled Kvasir-style reference image, without needing scikit-image.
def match_histograms_channel(source, reference):
    src_vals, src_idx, src_counts = np.unique(source.ravel(), return_inverse=True, return_counts=True)
    ref_vals, ref_counts = np.unique(reference.ravel(), return_counts=True)

    src_cdf = np.cumsum(src_counts).astype(np.float64)
    src_cdf /= src_cdf[-1]
    ref_cdf = np.cumsum(ref_counts).astype(np.float64)
    ref_cdf /= ref_cdf[-1]

    interp_vals = np.interp(src_cdf, ref_cdf, ref_vals)
    return interp_vals[src_idx].reshape(source.shape).astype(np.uint8)

def match_histograms_bgr(img_bgr, reference_bgr):
    matched = np.zeros_like(img_bgr)
    for c in range(3):
        matched[..., c] = match_histograms_channel(img_bgr[..., c], reference_bgr[..., c])
    return matched


# ── 3. Build a small target-domain reference pool (Kvasir-style images) ─────
# Uses a handful of Kvasir normal-cecum images purely as *style* references
# for histogram matching — NOT mixed into training labels/classes.
KVASIR_REF_DIR = r"D:\Marina\kvasir\kvasir-dataset-v2\normal-cecum"
N_REF_IMAGES = 20

reference_pool = []
if os.path.exists(KVASIR_REF_DIR):
    ref_candidates = glob.glob(os.path.join(KVASIR_REF_DIR, "*"))
    sampled_refs = random.sample(ref_candidates, min(N_REF_IMAGES, len(ref_candidates)))
    for ref_path in sampled_refs:
        ref_img = cv2.imread(ref_path)
        if ref_img is not None:
            reference_pool.append(ref_img)
    print(f"✅ Loaded {len(reference_pool)} reference images for histogram matching.")
else:
    print(f"⚠️  Kvasir reference dir not found at {KVASIR_REF_DIR} — "
          f"histogram matching will be skipped, CLAHE only.")


# ── 4. Generate domain-randomized augmented copies of the TRAIN split ───────
# Adds extra augmented versions alongside originals (does not overwrite them),
# so the model sees both VIM's native look and Kvasir-shifted variants.
AUG_FRACTION = 0.5        # fraction of train images to generate an augmented copy for
CLAHE_PROB = 0.7          # probability of applying CLAHE to a given augmented copy
HIST_MATCH_PROB = 0.5     # probability of applying histogram matching (if refs available)

YOLO_DIR = Path("vim_polyp_yolo/data/yolo/images")
train_dir = YOLO_DIR / 'train'
CLASS_NAMES = sorted([p.name for p in train_dir.iterdir() if p.is_dir()]) if train_dir.exists() else ANATOMICAL_ORDER
if not CLASS_NAMES:
    CLASS_NAMES = ANATOMICAL_ORDER

augmented_count = 0

for class_name in CLASS_NAMES:
    class_dir = train_dir / class_name
    if not class_dir.exists():
        continue

    image_paths = list(class_dir.glob('*.jpg'))
    n_to_augment = int(len(image_paths) * AUG_FRACTION)
    to_augment = random.sample(image_paths, min(n_to_augment, len(image_paths)))

    for img_path in tqdm(to_augment, desc=f"Augmenting {class_name}", leave=False):
        img = cv2.imread(str(img_path))
        if img is None:
            continue

        aug = img.copy()
        if random.random() < CLAHE_PROB:
            aug = apply_clahe(aug)
        if reference_pool and random.random() < HIST_MATCH_PROB:
            ref = random.choice(reference_pool)
            ref_resized = cv2.resize(ref, (aug.shape[1], aug.shape[0]))
            aug = match_histograms_bgr(aug, ref_resized)

        aug_path = img_path.parent / f"{img_path.stem}_domaug{img_path.suffix}"
        cv2.imwrite(str(aug_path), aug)
        augmented_count += 1

print(f"\n✅ Generated {augmented_count} domain-randomized augmented training images.")
print(f"   Train split size roughly increased by {AUG_FRACTION*100:.0f}%.")


# ── 5. Strengthen YOLO's built-in photometric augmentation for training ─────
# Widens hue/saturation/brightness jitter so the model can't overfit to
# VIM's fixed color calibration even on the non-augmented-copy images.
TRAIN_CFG['hsv_h'] = 0.03    # default 0.015 — hue jitter
TRAIN_CFG['hsv_s'] = 0.8     # default 0.7   — saturation jitter
TRAIN_CFG['hsv_v'] = 0.5     # default 0.4   — brightness/value jitter
TRAIN_CFG['auto_augment'] = 'randaugment'
TRAIN_CFG['erasing'] = 0.3   # random erasing, discourages fixating on fixed overlays/watermarks

print("\n✅ TRAIN_CFG updated with stronger color/illumination augmentation:")
for k in ['hsv_h', 'hsv_s', 'hsv_v', 'auto_augment', 'erasing']:
    print(f"   {k}: {TRAIN_CFG[k]}")

⚠️  Kvasir reference dir not found at D:\Marina\kvasir\kvasir-dataset-v2\normal-cecum — histogram matching will be skipped, CLAHE only.

✅ Generated 0 domain-randomized augmented training images.
   Train split size roughly increased by 50%.

✅ TRAIN_CFG updated with stronger color/illumination augmentation:
   hsv_h: 0.03
   hsv_s: 0.8
   hsv_v: 0.5
   auto_augment: randaugment
   erasing: 0.3


In [20]:
train_boundary_pair(*PAIR_LIST[0])   # cecum_vs_ascending

▶ cecum_vs_ascending
   ✅ cecum_vs_ascending: train=42, val=9, test=9 (balance='undersample', target/class=30)
WARNING AMP: checks failed . AMP training on NVIDIA GeForce GTX 1660 Ti GPU may cause NaN losses or zero-mAP results, so AMP will be disabled during training.
   ✅ done — train={'cecum': 21, 'ascending': 21}, val={'cecum': 5, 'ascending': 4}, test={'ascending': 5, 'cecum': 4}


YOLO(
  (model): ClassificationModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_

In [21]:
train_boundary_pair(*PAIR_LIST[1])   # ascending_vs_transverse

▶ ascending_vs_transverse
   ✅ ascending_vs_transverse: train=232, val=50, test=50 (balance='undersample', target/class=166)
WARNING AMP: checks failed . AMP training on NVIDIA GeForce GTX 1660 Ti GPU may cause NaN losses or zero-mAP results, so AMP will be disabled during training.
WARNING val: Slow image access detected (ping: 0.20.0 ms, read: 36.439.0 MB/s, size: 100.5 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
   ✅ done — train={'transverse': 116, 'ascending': 116}, val={'ascending': 25, 'transverse': 25}, test={'transverse': 25, 'ascending': 25}


YOLO(
  (model): ClassificationModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_

In [22]:
train_boundary_pair(*PAIR_LIST[2])   # transverse_vs_descending

▶ transverse_vs_descending
   ✅ transverse_vs_descending: train=232, val=50, test=50 (balance='undersample', target/class=166)
WARNING AMP: checks failed . AMP training on NVIDIA GeForce GTX 1660 Ti GPU may cause NaN losses or zero-mAP results, so AMP will be disabled during training.
WARNING val: Slow image access detected (ping: 0.10.0 ms, read: 7.14.0 MB/s, size: 110.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
   ✅ done — train={'transverse': 116, 'descending': 116}, val={'descending': 25, 'transverse': 25}, test={'transverse': 25, 'descending': 25}


YOLO(
  (model): ClassificationModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_

In [23]:
train_boundary_pair(*PAIR_LIST[3])   # descending_vs_sigmoid

▶ descending_vs_sigmoid
   ✅ descending_vs_sigmoid: train=3666, val=786, test=786 (balance='undersample', target/class=2619)
WARNING AMP: checks failed . AMP training on NVIDIA GeForce GTX 1660 Ti GPU may cause NaN losses or zero-mAP results, so AMP will be disabled during training.
   ✅ done — train={'sigmoid': 1833, 'descending': 1833}, val={'descending': 393, 'sigmoid': 393}, test={'descending': 393, 'sigmoid': 393}


YOLO(
  (model): ClassificationModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_

## 7. Evaluate Each Boundary Classifier

In [31]:
def evaluate_pairwise_model(model, dataset_dir, pair_name):
    val_results = model.val(split='test', verbose=False, data=str(dataset_dir))
    top1 = round(float(val_results.top1), 4)   # was val_results.metrics.top1
    return {'Pair': pair_name, 'Top-1 Accuracy': top1}

pairwise_eval_rows = []
for pair_name, model in pairwise_models.items():
    dataset_dir = pairwise_dataset_dirs[pair_name]
    try:
        pairwise_eval_rows.append(evaluate_pairwise_model(model, dataset_dir, pair_name))
        print(f"✅ {pair_name}: Top-1 = {pairwise_eval_rows[-1]['Top-1 Accuracy']:.3f}")
    except Exception as e:
        print(f"⚠️  Evaluation failed for {pair_name}: {e}")

df_pairwise_results = pd.DataFrame(pairwise_eval_rows)
df_pairwise_results.to_csv(RESULTS / 'pairwise_boundary_results.csv', index=False)
print("\n" + df_pairwise_results.to_string(index=False))


✅ cecum_vs_ascending: Top-1 = 0.889
✅ ascending_vs_transverse: Top-1 = 0.960
WARNING test: Slow image access detected (ping: 0.10.0 ms, read: 33.657.1 MB/s, size: 110.3 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
✅ transverse_vs_descending: Top-1 = 0.980
WARNING test: Slow image access detected (ping: 0.10.0 ms, read: 14.37.4 MB/s, size: 112.5 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
✅ descending_vs_sigmoid: Top-1 = 0.913

                    Pair  Top-1 Accuracy
      cecum_vs_ascending          0.8889
 ascending_vs_transverse          0.9600
transverse_vs_descending          0.9800
   descending_vs_sigmoid          0.9135


### Confusion matrix per boundary

Since each pair is a 2-class problem, this shows exactly which direction of confusion (e.g. mistaking ascending-colon frames for transverse, or vice versa) is more common for each boundary — the frames closest to the true anatomical transition are naturally the hardest for the model.

In [32]:
def pairwise_confusion_matrix(model, dataset_dir, class_a, class_b, n_samples=150):
    label_to_id = {class_a: 0, class_b: 1}
    y_true, y_pred = [], []

    test_dir = dataset_dir / 'test'
    candidates = []
    for cls_name in (class_a, class_b):
        cls_dir = test_dir / cls_name
        if cls_dir.exists():
            candidates.extend([(p, cls_name) for p in cls_dir.glob('*.jpg')])

    if not candidates:
        print(f"⚠️  No test images found for {class_a}_vs_{class_b}.")
        return

    sample = random.sample(candidates, min(n_samples, len(candidates)))
    for img_path, true_label in tqdm(sample, desc=f"{class_a}_vs_{class_b} eval", leave=False):
        res = model.predict(str(img_path), verbose=False)[0]
        if res.probs is None:
            continue
        pred_name = res.names[int(res.probs.top1)]
        if pred_name not in label_to_id:
            continue
        y_true.append(label_to_id[true_label])
        y_pred.append(label_to_id[pred_name])

    if not y_true:
        print(f"⚠️  No valid predictions for {class_a}_vs_{class_b}.")
        return

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[class_a, class_b])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{class_a} vs {class_b}', fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS / f'confusion_{class_a}_vs_{class_b}.png', dpi=150)
    plt.show()


for class_a, class_b in PAIR_LIST:
    pair_name = f"{class_a}_vs_{class_b}"
    if pair_name in pairwise_models:
        pairwise_confusion_matrix(pairwise_models[pair_name], pairwise_dataset_dirs[pair_name],
                                   class_a, class_b)


<Figure size 400x400 with 1 Axes>

<Figure size 400x400 with 1 Axes>

<Figure size 400x400 with 1 Axes>

<Figure size 400x400 with 1 Axes>

# Validation

kvasir-dataset-v2\normal-cecum

In [30]:
from collections import Counter
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO

# 1. Define paths for the pairwise model and dataset
PAIRWISE_WEIGHTS_PATH = r"D:\Marina\Nebras_RandD\runs\classify\vim_polyp_pairwise\results\cecum_vs_ascending-3\weights\best.pt"  # Update this!
DATASET_PATH = r"D:\Marina\kvasir\kvasir-dataset-v2\normal-cecum"

# 2. Check if weights file exists
if not os.path.exists(PAIRWISE_WEIGHTS_PATH):
    raise FileNotFoundError(
        f"Weights file not found at: {PAIRWISE_WEIGHTS_PATH}\n"
        "Please update `PAIRWISE_WEIGHTS_PATH` with the correct path to your binary model."
    )

# 3. Load the pairwise model
model = YOLO(PAIRWISE_WEIGHTS_PATH)

print("=" * 50)
print(f"Model Loaded. Classes: {model.names}")
print("=" * 50)

# 4. Run Inference (Note: imgsz=224 matches training resolution)
print("\n--- Running Pairwise Prediction ---")
results = model.predict(
    source=DATASET_PATH,
    imgsz=224,  # Matching 224x224 training resolution
    conf=0.25,
    save=True,
    project=r"D:\Marina\Nebras_RandD\runs\classify\kvasir_evaluation",
    name="pairwise_cecum_vs_ascending",
)

# 5. Extract Predictions
predictions = []
for result in results:
    top_class_id = result.probs.top1
    predicted_class_name = result.names[top_class_id]
    predictions.append(predicted_class_name)

# 6. Analyze Breakdown
pred_counts = Counter(predictions)
total_images = len(results)
all_classes = sorted(list(model.names.values()))

print("\n" + "=" * 50)
print(f"PAIRWISE BREAKDOWN FOR {total_images} 'NORMAL-CECUM' IMAGES")
print("=" * 50)

for cls_name in all_classes:
    count = pred_counts.get(cls_name, 0)
    pct = (count / total_images) * 100
    print(f"Predicted as '{cls_name}': {count:4d} / {total_images} ({pct:.2f}%)")

# Determine accuracy assuming ground truth is 'cecum' (or matching case)
cecum_key = next(
    (cls for cls in all_classes if "cecum" in cls.lower()), "cecum"
)
correct_cecum = pred_counts.get(cecum_key, 0)
accuracy = (correct_cecum / total_images) * 100

print("-" * 50)
print(
    f"Accuracy on Cecum folder: {accuracy:.2f}% ({correct_cecum}/{total_images} correct)"
)
print("=" * 50)

# 7. Plot Binary Confusion Matrix
cm_row = [pred_counts.get(cls, 0) for cls in all_classes]

plt.figure(figsize=(6, 3))
sns.heatmap(
    [cm_row],
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=all_classes,
    yticklabels=["True: cecum"],
    cbar=True,
)

plt.title(
    f"Pairwise Model Performance on Normal-Cecum Dataset (N={total_images})"
)
plt.xlabel("Predicted Label")
plt.ylabel("Ground Truth")
plt.tight_layout()
plt.show()

FileNotFoundError: Weights file not found at: D:\Marina\Nebras_RandD\runs\classify\vim_polyp_pairwise\results\cecum_vs_ascending-3\weights\best.pt
Please update `PAIRWISE_WEIGHTS_PATH` with the correct path to your binary model.

C3VD\cecum_t1_a

In [ ]:
from collections import Counter
import glob
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO

# 1. Define paths
WEIGHTS_PATH = r"D:\Marina\Nebras_RandD\runs\classify\vim_polyp_yolo\results\exp1_baseline\weights\best.pt"  # Or your pairwise model path
C3VD_FOLDER = r"D:\Marina\C3VD\cecum_t1_a"

# 2. Check path & filter for files ending with '_color.png'
color_images = glob.glob(os.path.join(C3VD_FOLDER, "*_color.png"))

if not os.path.exists(WEIGHTS_PATH):
    raise FileNotFoundError(f"Weights file not found at: {WEIGHTS_PATH}")

if len(color_images) == 0:
    # If images are inside a nested subfolder (e.g., cecum_t1_a/color/ or cecum_t1_a/rgb/)
    color_images = glob.glob(
        os.path.join(C3VD_FOLDER, "**", "*_color.png"), recursive=True
    )

print(
    f"Found {len(color_images)} color images ending with '_color.png' in {C3VD_FOLDER}"
)

if len(color_images) == 0:
    raise FileNotFoundError(
        "No files ending with '_color.png' were found. Please check your folder structure."
    )

# 3. Load model
model = YOLO(WEIGHTS_PATH)

# 4. Run Inference (matching training resolution imgsz=224)
print("\n--- Running Prediction on C3VD Color Images ---")
results = model.predict(
    source=color_images,  # Pass the explicit list of color image paths
    imgsz=224,  # Use 224x224 matching training resolution
    conf=0.25,
    save=True,
    project=r"D:\Marina\Nebras_RandD\runs\classify\c3vd_evaluation",
    name="cecum_t1_a_color_preds",
)

# 5. Extract Predictions
predictions = []
for result in results:
    top_class_id = result.probs.top1
    predicted_class_name = result.names[top_class_id]
    predictions.append(predicted_class_name)

# 6. Prediction Breakdown Analysis
total_images = len(results)
pred_counts = Counter(predictions)
all_classes = sorted(list(model.names.values()))

print("\n" + "=" * 50)
print(f"PREDICTION BREAKDOWN FOR C3VD CECUM_T1_A ({total_images} IMAGES)")
print("=" * 50)

for cls_name in all_classes:
    count = pred_counts.get(cls_name, 0)
    pct = (count / total_images) * 100
    print(f"Predicted as '{cls_name}': {count:4d} / {total_images} ({pct:.2f}%)")

# Determine accuracy assuming true anatomical region is 'cecum'
cecum_key = next(
    (cls for cls in all_classes if "cecum" in cls.lower()), "cecum"
)
correct_cecum = pred_counts.get(cecum_key, 0)
accuracy = (correct_cecum / total_images) * 100

print("-" * 50)
print(
    f"Accuracy on C3VD Cecum folder: {accuracy:.2f}% ({correct_cecum}/{total_images} correct)"
)
print("=" * 50)

# 7. Plot Prediction Distribution
cm_row = [pred_counts.get(cls, 0) for cls in all_classes]

plt.figure(figsize=(8, 3))
sns.heatmap(
    [cm_row],
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=all_classes,
    yticklabels=["True: cecum"],
    cbar=True,
)

plt.title(f"Model Predictions on C3VD cecum_t1_a Color Images (N={total_images})")
plt.xlabel("Predicted Class")
plt.ylabel("Ground Truth")
plt.tight_layout()
plt.show()